# Qwen2.5-0.5B From-Scratch Pipeline — Kaggle Runner

This notebook:
1. Installs the `qwenlab` package directly from the attached dataset (no file copying)
2. Runs the full test suite (39 tests — architecture equivalence vs real HF Qwen2/Qwen3, init, schedules, tasks, trainer)
3. Restores a checkpoint from a previous session, if one is attached (see "Resuming across sessions" below)
4. Runs the GPU feasibility smoke test (NOT real training yet — no real tokenizer/data wired in until the next phase)
5. Shows exactly where the trained checkpoint lives and how to load it back for inspection

**Before running:** Settings (right sidebar) → Accelerator → **GPU T4 x2 or P100**. This is a GPU feasibility test; running on CPU defeats the purpose.

## 1. Install qwenlab from the attached dataset

In [ ]:
import subprocess

result = subprocess.run(['find', '/kaggle/input', '-iname', 'pyproject.toml'], capture_output=True, text=True)
found = [p for p in result.stdout.strip().split('\n') if p]
assert found, "pyproject.toml not found under /kaggle/input -- confirm the qwenlab dataset is attached (Add Input, right sidebar)"
PKG_PATH = found[0].rsplit('/', 1)[0]
print("Found qwenlab package at:", PKG_PATH)

get_ipython().system(f'pip install -q -e "{PKG_PATH}"')

## 2. Run the full test suite

Should show `39 passed`. If this differs from the expected result, stop here and check the output before running anything else — better to catch a library-version difference now than after a long run.

In [ ]:
result = subprocess.run(['python', '-m', 'pytest', 'tests/', '-v'], cwd=PKG_PATH, capture_output=True, text=True)
print(result.stdout[-4000:])
print(result.stderr[-2000:])
assert result.returncode == 0, "Tests failed -- do not proceed to training until this passes"
print("\n✅ All tests passed.")

## 3. Resuming across sessions (read this before your first real training run)

A single Kaggle session maxes out at **12 hours**, but a real pretraining run will very likely need **multiple sessions**. The pattern:

- **End of a session:** commit this notebook (Save Version → Save & Run All). `/kaggle/working` becomes this notebook's permanent **Output**. From the Output tab, create a **new Kaggle Dataset** from that output (or a new version of an existing one) — call it e.g. `qwenlab-checkpoints`.
- **Start of the next session:** attach `qwenlab-checkpoints` as an **additional input** (alongside the `qwenlab` code dataset) via Add Input. The cell below automatically finds and restores it.

This is why the cell below searches `/kaggle/input` for a previous `progress.pt` rather than assuming a fixed path — it works whether this is session 1 (nothing found, trains from scratch) or session 5 (restores and continues).

In [ ]:
import shutil
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/qwenlab_run")
CKPT_DIR = OUTPUT_DIR / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

result = subprocess.run(['find', '/kaggle/input', '-iname', 'progress.pt'], capture_output=True, text=True)
found_ckpts = [p for p in result.stdout.strip().split('\n') if p]

if found_ckpts:
    src = found_ckpts[0]
    dst = CKPT_DIR / "progress.pt"
    shutil.copy2(src, dst)
    print(f"Restored checkpoint from previous session:\n  {src}\n  -> {dst}")
else:
    print("No previous checkpoint found under /kaggle/input -- this will be a fresh run (session 1).")

## 4. Run the GPU feasibility smoke test

Deliberately short (`TIME_BUDGET_MINUTES = 15`), not a real training run — this proves the real Qwen2.5-0.5B architecture fits and trains on this GPU, and that the time-budget/checkpoint mechanism works, before any real data exists. The actual multi-session pretraining run comes in the next phase, using this exact same resume mechanism with a much larger time budget.

In [ ]:
import sys
sys.path.insert(0, f"{PKG_PATH}/kaggle")
import smoke_test
print("smoke_test module loaded. TIME_BUDGET_MINUTES =", smoke_test.TIME_BUDGET_MINUTES)

In [ ]:
# Run with the shared OUTPUT_DIR so the resume-restore cell above and this run agree on the path
from qwenlab.config import QWEN2_5_0_5B, TrainConfig, InitConfig
from qwenlab.models.qwen import QwenForCausalLM, count_parameters
from qwenlab.init import apply_init
from qwenlab.training.trainer import Trainer
import torch, time

def main_with_output_dir():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    if device == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"Total GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

    cfg = QWEN2_5_0_5B
    cfg.vocab_size = smoke_test.VOCAB_SIZE_FOR_SMOKE_TEST
    model = QwenForCausalLM(cfg)
    apply_init(model, InitConfig(scheme="gpt2_scaled", std=0.02, seed=42))
    print(f"Parameters: {count_parameters(model):,}")

    train_cfg = TrainConfig(
        output_dir=str(OUTPUT_DIR), max_steps=10_000,
        micro_batch_size=smoke_test.BATCH_SIZE, grad_accum_steps=smoke_test.GRAD_ACCUM,
        lr=3e-4, warmup_steps=50, schedule="cosine", weight_decay=0.1, grad_clip=1.0,
        precision="auto", log_every=5, eval_every=100_000, save_every=20, keep_last=1,
        resume=True, time_budget_minutes=smoke_test.TIME_BUDGET_MINUTES,
    )
    trainer = Trainer(model, train_cfg, device=device)
    print(f"Precision: {trainer.dtype} | Resuming from step: {trainer.start_step}")

    batch_fn = smoke_test.synthetic_batch_fn(cfg.vocab_size, smoke_test.SEQ_LEN, smoke_test.BATCH_SIZE, device)

    def on_log(step, loss, lr, grad_norm):
        elapsed = time.time() - trainer.start_time
        toks = step * smoke_test.BATCH_SIZE * smoke_test.GRAD_ACCUM * smoke_test.SEQ_LEN
        mem = f"{torch.cuda.max_memory_allocated()/1e9:.2f} GB" if device == "cuda" else "n/a"
        print(f"step {step:5d} | loss {loss:.4f} | lr {lr:.2e} | {toks/max(elapsed,0.01):,.0f} tok/s | mem {mem} | {elapsed:.1f}s")

    result = trainer.fit(batch_fn, on_log=on_log)
    print(f"\nStopped at step {result['final_step']} ({'time budget reached' if result['stopped_early'] else 'max_steps reached'})")
    print(f"Checkpoint: {OUTPUT_DIR}/checkpoints/progress.pt")
    return result

smoke_result = main_with_output_dir()

## 5. Where the trained model is stored

**During/immediately after this session:** `/kaggle/working/qwenlab_run/checkpoints/progress.pt` (the resumable "latest" checkpoint) and periodic `step_N.pt` snapshots (last 2 kept) in the same folder.

**This does NOT survive past the session on its own.** To keep it:
1. **Save Version → Save & Run All (Commit)** — this re-runs the whole notebook and locks in `/kaggle/working` as the notebook's permanent Output.
2. From the Output tab, you can **download `progress.pt` directly**, or **create a new Dataset from it** (recommended — this is what the "Resuming across sessions" cell above looks for automatically next time).

**To check/use the trained model later** (this session or a future one), load it like this:

In [ ]:
import torch
from qwenlab.config import QWEN2_5_0_5B
from qwenlab.models.qwen import QwenForCausalLM

ckpt = torch.load(OUTPUT_DIR / "checkpoints" / "progress.pt", map_location="cpu", weights_only=False)
print("Saved at step:", ckpt["step"])

model = QwenForCausalLM(QWEN2_5_0_5B)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print("Loaded successfully -- model ready for inspection (e.g. a forward pass, or comparing weights against a later checkpoint).")

## 6. Commit this notebook

Save Version → **Save & Run All (Commit)**. Re-runs everything end to end and locks in the Output. After it completes, go to the Output tab and create/update the `qwenlab-checkpoints` dataset from `qwenlab_run/checkpoints/progress.pt`, ready for the next session.